In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("..")
RAW_PATH = BASE_DIR / "data" / "raw" / "2025_LoL_esports_match_data_from_OraclesElixir.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Loading:", RAW_PATH)

df = pd.read_csv(RAW_PATH, low_memory=False)

LEAGUES = ["LEC", "LCK", "LPL", "LCS", "PCS"]
df = df[df["league"].isin(LEAGUES)]

# Remove team summary rows (100, 200)
df = df[df["participantid"] <= 10]

# Sanity check
print(f"Total rows: {len(df)}")
print(f"\nMatches per league:")
print(df.groupby("league")["gameid"].nunique())
print(f"\nTotal unique matches: {df['gameid'].nunique()}")
print(f"\nRows per match (should be 10):")
print(df.groupby("gameid").size().value_counts())

Loading: ..\data\raw\2025_LoL_esports_match_data_from_OraclesElixir.csv
Total rows: 19690

Matches per league:
league
LCK    555
LEC    306
LPL    805
PCS    303
Name: gameid, dtype: int64

Total unique matches: 1969

Rows per match (should be 10):
10    1969
Name: count, dtype: int64


In [4]:
df = df[[
    "gameid", "league", "participantid", "side",
    "position", "playerid", "playername", "champion", "result"
]].copy()

needed = ["playerid", "champion", "result", "side", "league"]

df = df.dropna(subset=needed)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Map from elixir ids to numeric ids
player_encoder = LabelEncoder()
champ_encoder  = LabelEncoder()
league_encoder = LabelEncoder()

player_encoder.fit(df["playerid"])
champ_encoder.fit(df["champion"])
league_encoder.fit(df["league"])

# Transform strings to integers
df["player_id"] = player_encoder.transform(df["playerid"])
df["champ_id"]  = champ_encoder.transform(df["champion"])
df["league_id"] = league_encoder.transform(df["league"])

# Lookup tables for reversing id to name
playerid_to_name = df.drop_duplicates("playerid").set_index("playerid")["playername"]

player_lookup = pd.DataFrame({
    "player_id":   np.arange(len(player_encoder.classes_)),
    "playerid":    player_encoder.classes_,
    "player_name": [playerid_to_name[pid] for pid in player_encoder.classes_]
})

champ_lookup = pd.DataFrame({
    "champ_id":   np.arange(len(champ_encoder.classes_)),
    "champ_name": champ_encoder.classes_
})

league_lookup = pd.DataFrame({
    "league_id":   np.arange(len(league_encoder.classes_)),
    "league_name": league_encoder.classes_
})

# Save the lookups
player_lookup.to_csv(PROCESSED_DIR / "player_lookup.csv", index=False)
champ_lookup.to_csv(PROCESSED_DIR / "champ_lookup.csv", index=False)
league_lookup.to_csv(PROCESSED_DIR / "league_lookup.csv", index=False)

# Print basic stats
N_players = len(player_encoder.classes_)
N_champs  = len(champ_encoder.classes_)
N_leagues = len(league_encoder.classes_)

print(f"Players: {N_players}")
print(f"Champions: {N_champs}")
print(f"Leagues: {N_leagues} — {list(league_encoder.classes_)}")
print(f"\nSample player mapping:")
print(player_lookup.head(10))

Players: 316
Champions: 153
Leagues: 4 — ['LCK', 'LEC', 'LPL', 'PCS']

Sample player mapping:
   player_id                                   playerid player_name
0          0  oe:player:027a5217b659e8bedb70978dea494d6  Irrelevant
1          1  oe:player:03daae1844b9ad257324c3eedf41e28        Kiin
2          2  oe:player:051ca2e31ba25123961b3df95fdef4c     Delight
3          3  oe:player:05326a1c42e9c27a1067768c88955e7    Beichuan
4          4  oe:player:054e5f8aeb885db6713f4a3e7328efc         Jun
5          5  oe:player:060a269b0eb445a1b5075034787f0cc        Orca
6          6  oe:player:0be01924125adebc1a95bbb09d0c4d1      Starry
7          7  oe:player:0c74b7f78409a4022a2c4c5a5ca3ee1         369
8          8  oe:player:0d47af84bee7ca00a76beb860c3d6ff   Pretender
9          9  oe:player:0d9b0a3b3a93a8f759c9d8ac8eef97c     Breathe


In [ ]:
# Create lists to save
blue_players_list = [] # Who played on blue side
red_players_list  = [] # Who played on red side
blue_champs_list  = []
red_champs_list   = []
league_ids_list   = [] # The leagues
y_list            = [] # Outcomes (who won)
skipped = 0
skip_reasons = {"wrong_count": 0, "bad_sides": 0}

for gameid, group in df.groupby("gameid"):
    #Skip if a game doesnt have the correct number of entires
    if len(group) != 10:
        skip_reasons["wrong_count"] += 1
        skipped += 1
        continue
    # Sort players in order and which side they played in
    blue = group[group["side"] == "Blue"].sort_values("participantid")
    red  = group[group["side"] == "Red"].sort_values("participantid")

    # Skip bad data
    if len(blue) != 5 or len(red) != 5:
        skip_reasons["bad_sides"] += 1
        skipped += 1
        continue
    
    # Save the data we gathered
    blue_players_list.append(blue["player_id"].values)
    red_players_list.append(red["player_id"].values)
    blue_champs_list.append(blue["champ_id"].values)
    red_champs_list.append(red["champ_id"].values)
    league_ids_list.append(blue["league_id"].iloc[0])
    y_list.append(int(blue["result"].iloc[0]))

# Convert to arrays
blue_players = np.array(blue_players_list)
red_players  = np.array(red_players_list)
blue_champs  = np.array(blue_champs_list)
red_champs   = np.array(red_champs_list)
league_ids   = np.array(league_ids_list)
y            = np.array(y_list)

print(f"Skipped {skipped} games:")
for reason, count in skip_reasons.items():
    print(f"  {reason}: {count}")

print(f"\nFinal dataset: {len(y)} matches")
print(f"Blue winrate: {y.mean():.3f}")
print(f"\nMatches per league:")
for lid in range(N_leagues):
    lname = league_encoder.classes_[lid]
    count = (league_ids == lid).sum()
    print(f"  {lname}: {count}")

Skipped 9 games:
  wrong_count: 9
  bad_sides: 0

Final dataset: 1959 matches
Blue winrate: 0.523

Matches per league:
  LCK: 555
  LEC: 306
  LPL: 805
  PCS: 293


In [ ]:
# Save as npy files
np.save(PROCESSED_DIR / "blue_players.npy", blue_players)
np.save(PROCESSED_DIR / "red_players.npy",  red_players)
np.save(PROCESSED_DIR / "blue_champs.npy",  blue_champs)
np.save(PROCESSED_DIR / "red_champs.npy",   red_champs)
np.save(PROCESSED_DIR / "league_ids.npy",   league_ids)
np.save(PROCESSED_DIR / "y.npy",            y)

print("Saved to:", PROCESSED_DIR)
print("\nFiles:")
for f in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

print(f"\nDataset summary:")
print(f"  Matches:   {len(y)}")
print(f"  Players:   {N_players}")
print(f"  Champions: {N_champs}")
print(f"  Leagues:   {N_leagues}")

Saved to: ..\data\processed

Files:
  blue_champs.npy (76.6 KB)
  blue_players.npy (76.6 KB)
  champ_lookup.csv (1.7 KB)
  league_ids.npy (15.4 KB)
  league_lookup.csv (0.0 KB)
  player_lookup.csv (16.4 KB)
  red_champs.npy (76.6 KB)
  red_players.npy (76.6 KB)
  y.npy (15.4 KB)

Dataset summary:
  Matches:   1959
  Players:   316
  Champions: 153
  Leagues:   4
